New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import os, sys, json, nbformat
repo = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(repo / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(repo / 'benchmark/projects')
from ai_data_scientist import project_manager, lifecycle, language_router
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-04-tips')
project_manager.ensure_notebook(handle)
project_manager.ensure_data_dir(handle)
lifecycle.register_run('v020-exp-04', handle.notebook_path)
print({'notebook': str(handle.notebook_path), 'language': language_router.detect_language('チップ額とチップ率の特徴を日本語で分析してください'), 'cwd': str(Path.cwd())})

{'notebook': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-04-tips/notebooks/kaggle-v020-kaggle-exp-04-tips.ipynb', 'language': 'ja', 'cwd': '/home/nahisaho/GitHub/jupytermind'}


In [2]:
def append_cells(cells):
    lifecycle.mark_write_start('v020-exp-04')
    try:
        project_manager.enqueue_write(handle, lambda nb: nb.cells.extend(cells))
    finally:
        lifecycle.mark_write_end('v020-exp-04')

intro = '''# 飲食店 Tips：チップ額・チップ率に関係する特徴

run_id: `v020-exp-04`。説明・分析・APIアクセス・検証はJupyter MCPのPythonカーネル内で実行する。認証情報は表示しない。

## 分析計画
1. Kaggle SDKで認証、`tipping`を検索し、参照指定の`jsphyg/tipping`のファイル一覧からCSVを取得する。失敗した場合に限り旧実験で指定された公開CSVを使う。取得時刻とSHA-256でファイルを固定し、再実行時は同じファイルを照合する。
2. 元データの定義・分析前提をmanifest化し、列・型・欠損・重複・値域・カテゴリ・行間の意味的な整合性を調べる。同じ値の会計があり得るため、重複だけでは削除しない。
3. チップ額と `100 × tip / total_bill` を別々の目的変数にする。散布図、人数別・曜日別・時間帯別・喫煙区分別・記録上の性別の要約、Pearson/Spearman相関を確認する。
4. 金額・人数・カテゴリを同時に扱うOLS（HC3標準誤差）を探索的に用いる。効果の大きさと区間を重視し、カテゴリの薄い組合せや共線性を確認する。多重比較のため有意性を確証としない。
5. 分母共有の数理的影響、外れ値、線形性、曜日と時間帯の交絡を再検討する。感度分析と最大3回の自然言語による反復分析を記録し、価値のある未解決点がなくなれば終了する。
6. 全コードセルを新しいカーネルで上から再実行し、数値再現・`notebook_audit(visual_audit=True)`・lifecycleの終状態を確認する。

## 解釈の境界
観察データであり、無作為割付ではない。店・担当者・客の反復ID、サービス品質、地域、時期などが不明であるため、調整後の係数も因果効果ではない。ドル単位などの意味は確認できたものと推測を区別する。図中の割合は百分率（%）、差はパーセントポイント（pp）。
'''
setup = '''from pathlib import Path
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from datetime import datetime, timezone
from dataclasses import asdict
import os, sys, json, io, hashlib, zipfile, warnings
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display, Image
REPO = Path('/home/nahisaho/GitHub/jupytermind')
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(REPO / 'benchmark/projects')
from ai_data_scientist import project_manager, language_router, lifecycle
from ai_data_scientist import ingestion, eda, stats_analysis, data_definition
from ai_data_scientist import analysis_assumptions, data_quality, sensitivity
from ai_data_scientist import visualization, insight_engine, notebook_audit
RUN_ID = 'v020-exp-04'
SEED = 20261002
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-04-tips')
project_manager.ensure_notebook(handle)
data_dir = project_manager.ensure_data_dir(handle)
assert language_router.detect_language('チップ額とチップ率の関係を日本語で分析') == 'ja'
lifecycle.register_run(RUN_ID, handle.notebook_path)
event_log = []
@contextmanager
def phase(name, writing=False):
    if lifecycle.is_cancel_requested(RUN_ID):
        raise RuntimeError('実行キャンセル要求を検出')
    start = lifecycle.mark_write_start if writing else lifecycle.mark_execution_start
    end = lifecycle.mark_write_end if writing else lifecycle.mark_execution_end
    start(RUN_ID)
    event_log.append({'phase': name, 'event': 'start', 'at': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except Exception:
        lifecycle.mark_failed(RUN_ID)
        event_log.append({'phase': name, 'event': 'failed', 'at': datetime.now(timezone.utc).isoformat()})
        raise
    finally:
        end(RUN_ID)
        event_log.append({'phase': name, 'event': 'end', 'at': datetime.now(timezone.utc).isoformat()})
def write_notebook(mutation):
    with phase('notebook_write', writing=True):
        project_manager.enqueue_write(handle, mutation)
def emit(payload):
    display(json.dumps(payload, ensure_ascii=False, indent=2, default=str))
print('準備完了。固定乱数seed=', SEED)
'''
acquisition = '''# APIの例外本文/レスポンスヘッダ/認証オブジェクトは表示しない。
provenance_path = data_dir / 'provenance.json'
with phase('kaggle_acquisition'):
    if provenance_path.exists():
        provenance = json.loads(provenance_path.read_text(encoding='utf-8'))
        csv_path = data_dir / provenance['local_filename']
        assert hashlib.sha256(csv_path.read_bytes()).hexdigest() == provenance['sha256'], '固定データSHA-256不一致'
        acquisition_mode = '取得済みファイルをSHA-256検証して再利用（APIへの再問い合わせなし）'
    else:
        stage = 'import_authentication'
        api_failure = None
        try:
            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                from kaggle.api.kaggle_api_extended import KaggleApi
                api = KaggleApi()
                api.authenticate()
                stage = 'dataset_list'
                results = api.dataset_list(search='tipping')
                refs = [str(item.ref) for item in results]
                stage = 'dataset_list_files'
                listing = api.dataset_list_files('jsphyg/tipping')
                filenames = [item.name for item in listing.files]
                candidates = [name for name in filenames if name.lower().endswith('.csv')]
                if not candidates:
                    raise FileNotFoundError('対象CSVなし')
                remote_filename = next((name for name in candidates if Path(name).name.lower() == 'tips.csv'), candidates[0])
                stage = 'dataset_download_file'
                api.dataset_download_file('jsphyg/tipping', remote_filename, path=str(data_dir), force=True, quiet=True)
            csv_path = data_dir / Path(remote_filename).name
            if not csv_path.exists():
                archives = list(data_dir.glob('*.zip'))
                if len(archives) != 1:
                    raise FileNotFoundError('ダウンロード成果物を特定できない')
                with zipfile.ZipFile(archives[0]) as archive:
                    members = [name for name in archive.namelist() if Path(name).name == Path(remote_filename).name]
                    if len(members) != 1:
                        raise FileNotFoundError('アーカイブ内の対象CSVなし')
                    csv_path.write_bytes(archive.read(members[0]))
            source_kind = 'kaggle'
            source_url = 'https://www.kaggle.com/datasets/jsphyg/tipping'
        except Exception as exc:
            # HTTPコードと例外型のみ記録し、秘密が含まれ得る例外本文は保存しない。
            status = getattr(exc, 'status', None) or getattr(getattr(exc, 'response', None), 'status_code', None)
            api_failure = {'stage': stage, 'exception_type': type(exc).__name__, 'http_status': status,
                           'message': 'Kaggle API取得に失敗。認証情報保護のため例外本文を省略。'}
            legacy = Path('/home/nahisaho/kaggle/experiments/white-paper.md').read_text(encoding='utf-8')
            source_url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv'
            assert source_url in legacy, '旧実験の指定CSV URLを確認できない'
            import urllib.request
            with urllib.request.urlopen(source_url, timeout=45) as response:
                payload = response.read(2_000_001)
            if len(payload) > 2_000_000:
                raise ValueError('公開CSVサイズ上限超過')
            csv_path = data_dir / 'tips-fallback.csv'
            csv_path.write_bytes(payload)
            remote_filename = 'tips.csv'
            refs, filenames = [], []
            source_kind = 'public_csv_fallback'
        provenance = {'run_id': RUN_ID, 'source_kind': source_kind,
                      'requested_owner_dataset_slug': 'jsphyg/tipping',
                      'owner_dataset_slug': 'jsphyg/tipping' if source_kind == 'kaggle' else None,
                      'source_url': source_url, 'search': 'tipping', 'search_results': refs,
                      'listed_filenames': filenames, 'downloaded_filename': remote_filename,
                      'local_filename': csv_path.name, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
                      'sha256': hashlib.sha256(csv_path.read_bytes()).hexdigest(), 'api_failure': api_failure,
                      'fallback_reason': 'Kaggle API取得失敗のため、旧実験で指定された公開CSVを使用' if api_failure else None,
                      'identity_caveat': '公開CSVとKaggle掲載版の完全な同一性は保証されない' if api_failure else None}
        provenance_path.write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding='utf-8')
        acquisition_mode = '新規取得'
    loaded = ingestion.ingest(ingestion.SourceSpec('csv', str(csv_path)), row_limit=10000)
    assert not loaded.truncated
    raw = loaded.dataframe
    assert set(raw.columns) == {'total_bill','tip','sex','smoker','day','time','size'}, '想定と異なる列構成'
    df = raw.copy()
    emit({'acquisition_mode': acquisition_mode, 'provenance': provenance, 'shape': df.shape})
    display(df.head())
'''
append_cells([nbformat.v4.new_markdown_cell(intro), nbformat.v4.new_code_cell(setup, metadata={'analysis_id':'setup'}), nbformat.v4.new_markdown_cell('## 取得・ファイル同一性\n取得済みファイルは再実行時にハッシュ検証して利用する。別データで再実験する場合は別プロジェクトに取得する。'), nbformat.v4.new_code_cell(acquisition, metadata={'analysis_id':'acquisition'})])
print('分析計画・初期コードをenqueue_writeで保存しました。')

分析計画・初期コードをenqueue_writeで保存しました。


In [5]:
quality = '''with phase('definitions_quality'):
    FV = data_definition.FieldValue
    source = {'url': FV(provenance['source_url'], 'verified', '取得記録'),
              'sha256': FV(provenance['sha256'], 'verified', '取得ファイルのハッシュ'),
              'license': FV(None, 'unknown', 'CSVのみではライセンスを確認できない')}
    variables = {
      'total_bill': {'definition': FV('会計総額。チップを含むか・税の扱いは未確認', 'inferred', '列名'), 'unit': FV('米ドルと推測されるが本実験では辞書未確認', 'inferred', 'Tipsデータの通例')},
      'tip': {'definition': FV('チップ額', 'inferred', '列名・値'), 'unit': FV('total_billと同じ通貨と仮定', 'inferred', '値域')},
      'size': {'definition': FV('同席グループ人数', 'inferred', '列名・整数値'), 'unit': FV('人', 'inferred', '列名')},
      'sex': {'definition': FV('記録上のMale/Female。誰の属性か未確認', 'unknown', 'CSVには説明なし')},
      'smoker': {'definition': FV('記録上のYes/No。個人かグループか未確認', 'unknown', 'CSVには説明なし')},
      'day': {'definition': FV('曜日ラベル', 'inferred', 'Thur/Fri/Sat/Sun')},
      'time': {'definition': FV('Lunch/Dinner区分', 'inferred', '値')},
      'tip_rate_pct': {'definition': FV('100 * tip / total_bill。税・チップ含有不明なので列定義上の割合', 'verified', '本ノートブックの演算'), 'unit': FV('%', 'verified', '本ノートブックの演算')}
    }
    definition_manifest = data_definition.build_manifest(source,
      {'observation_unit': FV('1行は1会計と推測', 'inferred', '列構成'),
       'population': FV(None, 'unknown', '店・地域・期間・抽出方法不明'),
       'repeated_customer_server_id': FV(None, 'unknown', 'ID列なし')},
      variables, transformations=('全行維持', 'tip_rate_pct = 100 * tip / total_bill'))
    unknown_fields = [(path, asdict(value)) for path,value in definition_manifest.unresolved_fields()]
    inferred_fields = [(f'variables.{name}.{key}', asdict(value)) for name,fields in variables.items() for key,value in fields.items() if value.status == 'inferred']
    inferred_fields += [(f'dataset_scope.{key}', asdict(value)) for key,value in definition_manifest.dataset_scope.items() if value.status == 'inferred']
    schema = {'total_bill': {'min': 0.01, 'not_null': True}, 'tip': {'min': 0, 'not_null': True},
              'size': {'min': 1, 'not_null': True}, 'sex': {'allowed': ['Male','Female'], 'not_null': True},
              'smoker': {'allowed':['Yes','No'], 'not_null':True},
              'day': {'allowed':['Thur','Fri','Sat','Sun'], 'not_null': True},
              'time': {'allowed':['Lunch','Dinner'], 'not_null':True}}
    assert set(schema).issubset(df.columns)
    semantic_issues = data_quality.detect_anomalies(df, schema)
    assert not semantic_issues, '意味的制約違反。原因確認前にモデルへ進まない'
    assert np.isfinite(df[['total_bill','tip','size']].to_numpy(dtype=float)).all()
    assert (df['size'] % 1 == 0).all()
    df['tip_rate_pct'] = 100 * df.tip / df.total_bill
    quality_summary = {'rows':len(df), 'missing_cells':int(raw.isna().sum().sum()),
                       'exact_duplicate_rows':int(raw.duplicated().sum()),
                       'semantic_issues':[asdict(item) for item in semantic_issues],
                       'tip_exceeds_bill_rows':int((df.tip > df.total_bill).sum()),
                       'over_50_percent_tip_rows':df.loc[df.tip_rate_pct > 50, ['total_bill','tip','tip_rate_pct']].to_dict('records'),
                       'decision':'欠損・制約違反なし。重複値の会計は削除せず、外れ値も主解析に維持する'}
    assumption_manifest = analysis_assumptions.AnalysisAssumptionManifest(
      analysis_scope={'n':len(df), 'outcomes':['tip','tip_rate_pct'], 'model':'探索的OLS HC3', 'preprocessing':'削除・補完なし'},
      assumptions=(
        analysis_assumptions.Assumption('ratio_definition', 'tipとtotal_billは同通貨で、比率は列定義に基づく指標として解釈する', 'assumed', impact_if_false='チップ率の実務上の意味が変わる', conclusion_critical=True),
        analysis_assumptions.Assumption('observed_rows_only', '母集団の代表性を主張しない', 'verified'),
        analysis_assumptions.Assumption('independence', '会計単位の独立性。IDがなく検証不能', 'assumed', impact_if_false='HC3・bootstrapの区間が過小になる可能性', conclusion_critical=True),
        analysis_assumptions.Assumption('noncausal', '介入・識別戦略がなく因果効果を主張しない', 'verified'),
        analysis_assumptions.Assumption('duplicate_policy', '同値会計を残す。削除は感度分析のみ', 'assumed', impact_if_false='転記重複なら一部の重み付けが変わる')
      ), causal_scope='associational', sampling={'n':len(df),'seed':SEED})
    assumption_findings = analysis_assumptions.check_manifest(assumption_manifest)
    emit({'data_definition_manifest':asdict(definition_manifest), 'unknown_fields':unknown_fields,
          'inferred_fields':inferred_fields, 'quality':quality_summary,
          'assumption_manifest':asdict(assumption_manifest), 'assumption_findings':[asdict(f) for f in assumption_findings]})
    profile = eda.explore(raw)
    display(pd.DataFrame(profile.missing_summary).T)
    display(raw.describe(include='all').T)
    emit({'categorical_summary':profile.categorical_summary})
    print('適用不可: 独立参照データがないためvalidate_anomalies/compare_datasetsは実行しない。同系統CSVを独立検証とは扱わない。')
'''
descriptive = '''from scipy import stats
import statsmodels.formula.api as smf
with phase('descriptive_statistics'):
    base_metrics = {'n':len(df), 'mean_bill':float(df.total_bill.mean()), 'mean_tip':float(df.tip.mean()),
                    'mean_rate_pct':float(df.tip_rate_pct.mean()), 'median_rate_pct':float(df.tip_rate_pct.median()),
                    'bill_weighted_rate_pct':float(100*df.tip.sum()/df.total_bill.sum())}
    corr_rows = []
    for outcome in ['tip','tip_rate_pct']:
        for feature in ['total_bill','size']:
            result = stats_analysis.correlation(df,feature,outcome,language='ja')
            rho,p = stats.spearmanr(df[feature],df[outcome])
            corr_rows.append({'feature':feature,'outcome':outcome,'Pearson_r':result.statistic,
                              'Spearman_rho':float(rho),'Pearson_p':result.p_value,'interpretation':result.interpretation})
    correlations = pd.DataFrame(corr_rows)
    group_tables = {}
    for col in ['size','day','time','smoker','sex']:
        table = df.groupby(col,observed=True).agg(n=('tip','size'),mean_bill=('total_bill','mean'),mean_tip=('tip','mean'),
                                                mean_rate_pct=('tip_rate_pct','mean'),median_rate_pct=('tip_rate_pct','median'),
                                                sum_tip=('tip','sum'),sum_bill=('total_bill','sum'))
        table['bill_weighted_rate_pct'] = 100 * table.sum_tip / table.sum_bill
        table = table.drop(columns=['sum_tip','sum_bill'])
        group_tables[col] = table
        print('グループ:',col)
        display(table.round(3))
    emit({'base_metrics':base_metrics,'correlations':corr_rows})
    display(pd.crosstab(df.day, df.time))
'''
model = '''with phase('adjusted_models'):
    formula = 'total_bill + size + C(day) + C(time) + C(smoker) + C(sex)'
    fits = {outcome:smf.ols(outcome+' ~ '+formula, data=df).fit(cov_type='HC3') for outcome in ['tip','tip_rate_pct']}
    def coefficient_table(fit):
        ci = fit.conf_int()
        return pd.DataFrame({'coefficient':fit.params,'HC3_se':fit.bse,'CI95_low':ci[0],'CI95_high':ci[1],'exploratory_p':fit.pvalues})
    for outcome,fit in fits.items():
        print('目的変数:',outcome,'R2=',round(fit.rsquared,4))
        display(coefficient_table(fit).round(5))
    X = fits['tip'].model.exog
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    vif = pd.DataFrame({'term':fits['tip'].model.exog_names,'VIF':[variance_inflation_factor(X,i) for i in range(X.shape[1])]})
    display(vif.round(3))
    initial_metrics = {**base_metrics,
      'bill_tip_pearson':float(correlations.loc[(correlations.feature=='total_bill')&(correlations.outcome=='tip'),'Pearson_r'].iloc[0]),
      'bill_rate_pearson':float(correlations.loc[(correlations.feature=='total_bill')&(correlations.outcome=='tip_rate_pct'),'Pearson_r'].iloc[0]),
      'bill_rate_spearman':float(correlations.loc[(correlations.feature=='total_bill')&(correlations.outcome=='tip_rate_pct'),'Spearman_rho'].iloc[0]),
      'adjusted_tip_bill_coef':float(fits['tip'].params['total_bill']),
      'adjusted_tip_size_coef':float(fits['tip'].params['size']),
      'adjusted_rate_bill_coef':float(fits['tip_rate_pct'].params['total_bill']),
      'design_rank':int(np.linalg.matrix_rank(X)), 'design_columns':X.shape[1],
      'max_leverage':float(fits['tip'].get_influence().hat_matrix_diag.max()),
      'thursday_dinner_n':int(((df.day=='Thur')&(df.time=='Dinner')).sum())}
    emit({'initial_metrics':initial_metrics})
'''
chart = '''import matplotlib.pyplot as plt
with phase('initial_visualizations'):
    # render_chartに日本語ラベルを渡し、日本語フォントを初期化する。
    png = visualization.render_chart(df, kind='scatter', x='total_bill', y='tip',
        title='会計総額とチップ額（244会計）',xlabel='会計総額（原データ通貨単位）',ylabel='チップ額（原データ通貨単位）')
    display(visualization.build_image_output(png)['data'],raw=True)
    fig, axes = plt.subplots(2,2, figsize=(12,9), constrained_layout=True)
    a=axes[0,0]
    for label,sub in df.groupby('smoker'):
        a.scatter(sub.total_bill,sub.tip_rate_pct,s=24,alpha=.6,label='喫煙 '+label)
    a.set(title='会計総額とチップ率：分母共有に注意',xlabel='会計総額（原データ通貨単位）',ylabel='チップ率（%）'); a.legend()
    a=axes[0,1]
    table=group_tables['size']; pos=np.arange(len(table))
    a.bar(pos,table.mean_tip,color='#4477AA',label='平均チップ額')
    a.set(xticks=pos,xticklabels=[f'{k}人\\nn={int(n)}' for k,n in zip(table.index,table.n)],title='人数と平均チップ額（少数群に注意）',xlabel='グループ人数',ylabel='チップ額（原データ通貨単位）');a.legend()
    a=axes[1,0]
    order=['Thur','Fri','Sat','Sun']; values=[df.loc[df.day==day,'tip_rate_pct'] for day in order]
    a.boxplot(values, tick_labels=[f'{day}\\nn={len(v)}' for day,v in zip(order,values)],showfliers=True)
    a.set(title='曜日別チップ率の分布',xlabel='曜日',ylabel='チップ率（%）')
    a=axes[1,1]
    tab=group_tables['time']; pos=np.arange(len(tab))
    a.bar(pos-.16,tab.mean_rate_pct,width=.32,label='会計ごとの平均')
    a.bar(pos+.16,tab.bill_weighted_rate_pct,width=.32,label='金額加重（総チップ/総会計）')
    a.set(xticks=pos,xticklabels=[f'{k}\\nn={int(n)}' for k,n in zip(tab.index,tab.n)],title='時間帯別：2種類の率を区別',xlabel='時間帯',ylabel='チップ率（%）');a.legend()
    with warnings.catch_warnings(record=True) as observed:
        warnings.simplefilter('always')
        buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=140)
    plt.close(fig)
    glyph_warnings=[str(w.message) for w in observed if 'Glyph' in str(w.message)]
    assert not glyph_warnings, '図の文字欠落を検出'
    display(visualization.build_image_output(buffer.getvalue())['data'],raw=True)
    emit({'visual_check':{'missing_glyph_warnings':glyph_warnings,'layout':'2x2 constrained_layout','labels':'タイトル/軸/単位/凡例/群nを確認'}})
'''
# API検索ログで未保持の結果を空の検索結果と誤解させないよう訂正する。
if provenance['source_kind']=='public_csv_fallback' and provenance['api_failure']['stage']=='dataset_list_files':
    provenance['search_results']=None
    provenance['search_status']='dataset_list(search="tipping")成功。初回例外経路で結果一覧を保持しなかったため一覧は不明。'
    provenance_path.write_text(json.dumps(provenance,ensure_ascii=False,indent=2),encoding='utf-8')
def preserve_search(nb):
    c=next(c for c in nb.cells if c.metadata.get('analysis_id')=='acquisition')
    c.source=c.source.replace("api_failure = None", "api_failure = None\n        refs, filenames = [], []")
    c.source=c.source.replace("            refs, filenames = [], []\n            source_kind", "            source_kind")
project_manager.enqueue_write(handle,preserve_search)
append_cells([
    nbformat.v4.new_markdown_cell('## データ定義manifest・分析前提manifest・意味的検査\n本実験はCSVのみを確認した。意味の推測・独立性の未検証を明示し、辞書未確認の単位を確定事実としない。'),
    nbformat.v4.new_code_cell(quality,metadata={'analysis_id':'quality'}),
    nbformat.v4.new_markdown_cell('## 初回分析：記述統計・相関\n平均率と金額加重率は異なる推定対象である。Pearsonは線形、Spearmanは順位関係を調べる。p値は会計の独立性を仮定した探索的指標。'),
    nbformat.v4.new_code_cell(descriptive,metadata={'analysis_id':'descriptive'}),
    nbformat.v4.new_markdown_cell('## 調整後の関連\n基準群はFri、Dinner、No、Female。連続変数は1単位増加、カテゴリは基準群との差。HC3は不均一分散に対応するが、観測間依存や未観測交絡を解決しない。'),
    nbformat.v4.new_code_cell(model,metadata={'analysis_id':'models'}),
    nbformat.v4.new_markdown_cell('## 図表：金額・割合・人数・曜日・時間帯\n箱ひげ図は中央値と四分位範囲、点は外れ値を含む会計。人数5/6など小標本の平均は特に不安定。金額は通貨辞書未確認のため原データ通貨単位と表示する。'),
    nbformat.v4.new_code_cell(chart,metadata={'analysis_id':'charts_initial','chart':{'title':'金額と率・人数・曜日・時間帯','xlabel':'会計総額/グループ人数/曜日/時間帯','ylabel':'チップ額/チップ率（%）','legend':'喫煙Yes/No・平均額・会計平均率/金額加重率；箱ひげ図は単一系列','missing_glyphs':[]}})
])
print('初回分析コードを保存しました。')

初回分析コードを保存しました。


In [11]:
request1 = '会計総額が高いほどチップ率が低いという結論は、71%の1会計や比率の分母共有に左右されていませんか。高チップ率・会計額の両端・重複を除く仕様と、夕食のみ・人数2〜4人のみで感度分析し、対数モデルの弾力性と加法モデルから生じる率の曲線を使って、観察上の関連と行動・因果の解釈を切り分けてください。'
iteration1 = '''with phase('iteration_1_sensitivity'):
    subsets = {
      'all':df,
      'rate_below_p99':df.loc[df.tip_rate_pct <= df.tip_rate_pct.quantile(.99)],
      'bill_central_90pct':df.loc[df.total_bill.between(df.total_bill.quantile(.05),df.total_bill.quantile(.95))],
      'drop_exact_duplicates':df.loc[~raw.duplicated()],
      'dinner_only':df.loc[df.time=='Dinner'],
      'size_2_to_4':df.loc[df['size'].between(2,4)]}
    def slope(subset, outcome):
        return float(smf.ols(outcome+' ~ '+formula,data=subsets[subset]).fit().params['total_bill'])
    sensitivity_reports = {}
    sensitivity_rows=[]
    for outcome in ['tip','tip_rate_pct']:
        plan = sensitivity.SensitivityPlan(parameter_grid={'subset':list(subsets),'outcome':[outcome]},max_runs=6)
        report = sensitivity.run_sensitivity(plan,slope,stability_tolerance=.25)
        sensitivity_reports[outcome]=report
        for result in report.results:
            sensitivity_rows.append({**result.specification,'n':len(subsets[result.specification['subset']]),'bill_coefficient':result.value})
    sensitivity_table = pd.DataFrame(sensitivity_rows)
    display(sensitivity_table.round(5))
    assert (df.tip > 0).all()
    log_fit = smf.ols('np.log(tip) ~ np.log(total_bill) + size + C(day) + C(time) + C(smoker) + C(sex)',data=df).fit(cov_type='HC3')
    elasticity = float(log_fit.params['np.log(total_bill)'])
    elasticity_ci = log_fit.conf_int().loc['np.log(total_bill)'].tolist()
    additive_fit = smf.ols('tip ~ total_bill',data=df).fit(cov_type='HC3')
    intercept,linear_slope = additive_fit.params[['Intercept','total_bill']]
    # E[tip|bill] = a + b*bill なら、対応する率 = 100*(a/bill+b)。
    implied_rate_10 = float(100*(intercept/10+linear_slope))
    implied_rate_40 = float(100*(intercept/40+linear_slope))
    iteration1_metrics = {'elasticity':elasticity,'elasticity_CI95':elasticity_ci,
      'additive_intercept':float(intercept),'additive_slope':float(linear_slope),
      'implied_rate_at_bill_10_pct':implied_rate_10,'implied_rate_at_bill_40_pct':implied_rate_40,
      'tip_sensitivity_stable':sensitivity_reports['tip'].stable,
      'tip_max_relative_deviation':sensitivity_reports['tip'].max_relative_deviation,
      'rate_sensitivity_stable':sensitivity_reports['tip_rate_pct'].stable,
      'rate_max_relative_deviation':sensitivity_reports['tip_rate_pct'].max_relative_deviation,
      'all_tip_slopes_positive':bool((sensitivity_table.loc[sensitivity_table.outcome=='tip','bill_coefficient']>0).all()),
      'all_rate_slopes_negative':bool((sensitivity_table.loc[sensitivity_table.outcome=='tip_rate_pct','bill_coefficient']<0).all())}
    emit({'iteration1_metrics':iteration1_metrics,'stability_tolerance':.25,'sensitivity_runs_per_outcome':6,
          'interpretation':'同じ向きかと係数の大きさの安定性を別々に評価する。部分集合は推定対象も変える。加法モデルの曲線は数理的説明で、心理・因果を識別しない。'})
    fig,axes=plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
    a=axes[0];a.scatter(df.total_bill,df.tip_rate_pct,s=18,alpha=.4,label='全244会計（外れ値も表示）')
    bill_grid=np.linspace(df.total_bill.min(),df.total_bill.max(),200)
    a.plot(bill_grid,100*(intercept/bill_grid+linear_slope),color='crimson',label='チップ額の加法モデルから導いた率')
    a.set(title='分母共有：正の定額成分だけでも率は下がる',xlabel='会計総額（原データ通貨単位）',ylabel='チップ率（%）');a.legend(fontsize=8)
    a=axes[1]
    view=df.loc[df.tip_rate_pct<=40]
    for label,sub in view.groupby('smoker'):
        a.scatter(sub.total_bill,sub.tip_rate_pct,s=22,alpha=.6,label='喫煙 '+label)
    a.set(title=f'中心部分の拡大（40%超の{len(df)-len(view)}会計は左図に表示）',xlabel='会計総額（原データ通貨単位）',ylabel='チップ率（%）');a.set_ylim(0,40);a.legend()
    with warnings.catch_warnings(record=True) as observed:
        warnings.simplefilter('always');buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=140)
    plt.close(fig)
    assert not [w for w in observed if 'Glyph' in str(w.message)]
    display(visualization.build_image_output(buffer.getvalue())['data'],raw=True)
'''
append_cells([
 nbformat.v4.new_markdown_cell('## 反復依頼履歴：反復1\n\n### 追加依頼文（原文）\n'+request1+'\n\n### 選定理由\n初回の率との相関は−0.339だが、71%の会計と分母共有がある。主結論の向き・大きさ・説明を変える可能性があり優先する。証拠は初回models/quality/descriptiveセル、追加結果はiteration1セル。'),
 nbformat.v4.new_code_cell(iteration1,metadata={'analysis_id':'iteration1','chart':{'title':'加法モデルによる率・中心部分の拡大','xlabel':'会計総額（原データ通貨単位）','ylabel':'チップ率（%）','legend':'全会計/加法モデル、喫煙Yes/No','missing_glyphs':[]}})
])
print('反復1の依頼文とコードを保存しました。')

反復1の依頼文とコードを保存しました。


In [13]:
request2 = '曜日・昼夜・喫煙区分・記録上の性別に関係がないと言い切れますか。群ごとのチップ率差にbootstrap区間を付け、会計平均率と金額加重率で喫煙区分の比較が反転する理由を確認してください。曜日と時間帯の重なり不足を調べ、同じ曜日の昼夜比較と曜日・時間帯を片方ずつ入れたモデルで、識別できない関係を明確にしてください。'
iteration2 = '''with phase('iteration_2_group_uncertainty'):
    rng=np.random.default_rng(SEED)
    B=5000
    def bootstrap_difference(frame, group, positive, negative, measure='mean'):
        pos=frame.loc[frame[group]==positive,['tip','total_bill','tip_rate_pct']].to_numpy()
        neg=frame.loc[frame[group]==negative,['tip','total_bill','tip_rate_pct']].to_numpy()
        assert len(pos)>1 and len(neg)>1
        p=pos[rng.integers(0,len(pos),size=(B,len(pos)))]
        n=neg[rng.integers(0,len(neg),size=(B,len(neg)))]
        if measure=='mean':
            samples=p[:,:,2].mean(axis=1)-n[:,:,2].mean(axis=1)
            estimate=float(pos[:,2].mean()-neg[:,2].mean())
        else:
            samples=100*p[:,:,0].sum(axis=1)/p[:,:,1].sum(axis=1)-100*n[:,:,0].sum(axis=1)/n[:,:,1].sum(axis=1)
            estimate=float(100*pos[:,0].sum()/pos[:,1].sum()-100*neg[:,0].sum()/neg[:,1].sum())
        low,high=np.quantile(samples,[.025,.975])
        return {'contrast':f'{group}: {positive} - {negative}','measure':measure,'n_positive':len(pos),'n_negative':len(neg),
                'difference_pp':estimate,'CI95_low_pp':float(low),'CI95_high_pp':float(high)}
    contrasts=[bootstrap_difference(df,'smoker','Yes','No'),
               bootstrap_difference(df,'smoker','Yes','No','bill_weighted'),
               bootstrap_difference(df,'sex','Male','Female'),
               bootstrap_difference(df,'time','Lunch','Dinner'),
               bootstrap_difference(df.loc[df.day=='Fri'],'time','Lunch','Dinner')]
    contrast_table=pd.DataFrame(contrasts)
    display(contrast_table.round(4))
    time_specs=[]
    for name,formula2 in [('day_and_time',formula),('time_only','total_bill + size + C(time) + C(smoker) + C(sex)'),
                           ('day_only','total_bill + size + C(day) + C(smoker) + C(sex)')]:
        fit=smf.ols('tip_rate_pct ~ '+formula2,data=df).fit(cov_type='HC3')
        term='C(time)[T.Lunch]'
        if term in fit.params:
            low,high=fit.conf_int().loc[term]
            time_specs.append({'specification':name,'Lunch_minus_Dinner_pp':float(fit.params[term]),'CI95_low_pp':float(low),'CI95_high_pp':float(high)})
    display(pd.DataFrame(time_specs).round(4))
    cov_by_smoking=[]
    for label,sub in df.groupby('smoker'):
        cov=float(np.mean((sub.total_bill-sub.total_bill.mean())*(sub.tip_rate_pct-sub.tip_rate_pct.mean())))
        identity=float(sub.tip_rate_pct.mean()+cov/sub.total_bill.mean())
        cov_by_smoking.append({'smoker':label,'mean_rate_pct':float(sub.tip_rate_pct.mean()),
              'weighted_rate_pct':float(100*sub.tip.sum()/sub.total_bill.sum()),'bill_rate_covariance':cov,
              'identity_mean_plus_cov_over_bill':identity,'bill_rate_Spearman':float(stats.spearmanr(sub.total_bill,sub.tip_rate_pct).statistic)})
    display(pd.DataFrame(cov_by_smoking).round(5))
    iteration2_metrics={'smoker_mean_difference_pp':contrasts[0]['difference_pp'],
      'smoker_weighted_difference_pp':contrasts[1]['difference_pp'],
      'smoker_mean_CI95':[contrasts[0]['CI95_low_pp'],contrasts[0]['CI95_high_pp']],
      'Friday_lunch_dinner_difference_pp':contrasts[4]['difference_pp'],
      'Friday_lunch_dinner_CI95':[contrasts[4]['CI95_low_pp'],contrasts[4]['CI95_high_pp']],
      'time_VIF':float(vif.loc[vif.term=='C(time)[T.Lunch]','VIF'].iloc[0]),
      'smoker_Yes_bill_rate_Spearman':next(r['bill_rate_Spearman'] for r in cov_by_smoking if r['smoker']=='Yes'),
      'smoker_No_bill_rate_Spearman':next(r['bill_rate_Spearman'] for r in cov_by_smoking if r['smoker']=='No')}
    emit({'iteration2_metrics':iteration2_metrics,'bootstrap_repetitions':B,'seed':SEED,
      'limitation':'会計を独立に再標本化するpercentile区間。多数の探索的比較に未調整。CIが0を含むことは同等性・無効果の証明ではない。金額加重率は大きな会計の重みが増える別指標。'})
    fig,axes=plt.subplots(1,2,figsize=(12,4.8),constrained_layout=True)
    a=axes[0];pos=np.arange(len(contrast_table));m=contrast_table.difference_pp.to_numpy()
    errors=np.vstack([m-contrast_table.CI95_low_pp.to_numpy(),contrast_table.CI95_high_pp.to_numpy()-m])
    a.errorbar(m,pos,xerr=errors,fmt='o',capsize=4,label='点推定と95% bootstrap区間')
    a.axvline(0,color='gray',ls='--',label='差ゼロ')
    a.set(yticks=pos,yticklabels=['喫煙 Yes−No：会計平均','喫煙 Yes−No：金額加重','Male−Female：会計平均','Lunch−Dinner：全曜日','Lunch−Dinner：金曜のみ'],title='群差と不確実性（因果差ではない）',xlabel='チップ率差（pp）',ylabel='比較');a.legend(fontsize=8,loc='lower right')
    a=axes[1];cross=pd.crosstab(df.day,df.time).reindex(['Thur','Fri','Sat','Sun'])
    im=a.imshow(cross.to_numpy(),cmap='Blues',vmin=0)
    for (i,j),value in np.ndenumerate(cross.to_numpy()):a.text(j,i,str(value),ha='center',va='center',color='white' if value>45 else 'black')
    a.set(xticks=[0,1],xticklabels=cross.columns,yticks=range(4),yticklabels=cross.index,title='曜日×時間帯：比較可能な重なりが乏しい',xlabel='時間帯',ylabel='曜日')
    fig.colorbar(im,ax=a,label='会計数 n')
    with warnings.catch_warnings(record=True) as observed:
        warnings.simplefilter('always');buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=140)
    plt.close(fig)
    assert not [w for w in observed if 'Glyph' in str(w.message)]
    display(visualization.build_image_output(buffer.getvalue())['data'],raw=True)
'''
append_cells([
 nbformat.v4.new_markdown_cell('## 反復依頼履歴：反復2\n\n### 追加依頼文（原文）\n'+request2+'\n\n### 選定理由\n反復1で額の係数は安定だが率の大きさは不安定。初回の喫煙群比較は平均率と金額加重率で逆向きであり、昼夜のVIFも約9.24。無関係と誤解する危険を解消する。証拠はdescriptive/models/iteration1セル、追加結果はiteration2セル。'),
 nbformat.v4.new_code_cell(iteration2,metadata={'analysis_id':'iteration2','chart':{'title':'群差区間・曜日時間帯クロス表','xlabel':'差（pp）/時間帯','ylabel':'比較/曜日','legend':'点推定・95%区間・差ゼロ・会計数カラー尺度','missing_glyphs':[]}})
])
print('反復2を保存しました。')

反復2を保存しました。


In [15]:
request3 = '喫煙Yes群では会計総額とチップ率の順位相関がNo群より負に強いようです。一定の群差だけを仮定する初回モデルで見落としていませんか。会計総額×喫煙区分の交互作用をチップ額とチップ率の両方で探索し、高率の会計除外・会計額中央90%・夕食のみの仕様で確認してください。比較範囲の重なりと少数群も示し、探索的な候補と安定した結論を区別してください。'
iteration3 = '''with phase('iteration_3_interaction'):
    selected=['all','rate_below_p99','bill_central_90pct','dinner_only']
    interaction_formula = 'total_bill * C(smoker) + size + C(day) + C(time) + C(sex)'
    term='total_bill:C(smoker)[T.Yes]'
    interaction_rows=[]
    for outcome in ['tip','tip_rate_pct']:
        for name in selected:
            fit=smf.ols(outcome+' ~ '+interaction_formula,data=subsets[name]).fit(cov_type='HC3')
            low,high=fit.conf_int().loc[term]
            interaction_rows.append({'outcome':outcome,'subset':name,'n':len(subsets[name]),
              'Yes_minus_No_bill_slope':float(fit.params[term]),'CI95_low':float(low),'CI95_high':float(high),
              'bill_slope_No':float(fit.params['total_bill']),
              'bill_slope_Yes':float(fit.params['total_bill']+fit.params[term])})
    interaction_table=pd.DataFrame(interaction_rows)
    display(interaction_table.round(5))
    interaction_reports={}
    def interaction_coefficient(subset,outcome):
        return float(interaction_table.loc[(interaction_table['subset']==subset)&(interaction_table.outcome==outcome),'Yes_minus_No_bill_slope'].iloc[0])
    for outcome in ['tip','tip_rate_pct']:
        plan=sensitivity.SensitivityPlan(parameter_grid={'subset':selected,'outcome':[outcome]},max_runs=4)
        interaction_reports[outcome]=sensitivity.run_sensitivity(plan,interaction_coefficient,stability_tolerance=.25)
    range_table=df.groupby('smoker').total_bill.agg(['min','max','count'])
    overlap=[float(range_table['min'].max()),float(range_table['max'].min())]
    display(range_table)
    binned=pd.crosstab(pd.cut(df.total_bill,bins=[0,10,20,30,60],right=False),df.smoker)
    display(binned)
    full_rate=interaction_table.loc[(interaction_table.outcome=='tip_rate_pct')&(interaction_table['subset']=='all')].iloc[0]
    iteration3_metrics={'rate_interaction_slope_difference':float(full_rate.Yes_minus_No_bill_slope),
      'rate_interaction_CI95':[float(full_rate.CI95_low),float(full_rate.CI95_high)],
      'tip_interaction_stable':interaction_reports['tip'].stable,
      'tip_interaction_max_relative_deviation':interaction_reports['tip'].max_relative_deviation,
      'rate_interaction_stable':interaction_reports['tip_rate_pct'].stable,
      'rate_interaction_max_relative_deviation':interaction_reports['tip_rate_pct'].max_relative_deviation,
      'bill_common_range':overlap,
      'rate_interaction_CI_excludes_zero_all_specs':bool(((interaction_table.loc[interaction_table.outcome=='tip_rate_pct','CI95_high']<0)|(interaction_table.loc[interaction_table.outcome=='tip_rate_pct','CI95_low']>0)).all())}
    emit({'iteration3_metrics':iteration3_metrics,'stability_tolerance':.25,
      'limitation':'データを見てから選んだ交互作用。選択・多重比較を補正した確証ではない。範囲の重なりは他の共変量の交換可能性を保証しない。区間は独立会計を仮定する。'})
    fig,axes=plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
    labels=['全会計','高率上位1%除外','会計中央90%','夕食のみ']
    for a,outcome,title,unit in zip(axes,['tip','tip_rate_pct'],['チップ額：喫煙区分による傾きの差','チップ率：喫煙区分による傾きの差'],['通貨単位/会計通貨単位','pp/会計通貨単位']):
        table=interaction_table.loc[interaction_table.outcome==outcome];m=table.Yes_minus_No_bill_slope.to_numpy();pos=np.arange(4)
        err=np.vstack([m-table.CI95_low.to_numpy(),table.CI95_high.to_numpy()-m])
        a.errorbar(m,pos,xerr=err,fmt='o',capsize=4,label='調整後の差と95% HC3区間');a.axvline(0,color='gray',ls='--',label='傾きの差ゼロ')
        a.set(yticks=pos,yticklabels=labels,title=title,xlabel='Yes−Noの傾き差（'+unit+'）',ylabel='分析仕様');a.legend(fontsize=8)
    with warnings.catch_warnings(record=True) as observed:
        warnings.simplefilter('always');buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=140)
    plt.close(fig)
    assert not [w for w in observed if 'Glyph' in str(w.message)]
    display(visualization.build_image_output(buffer.getvalue())['data'],raw=True)
'''
append_cells([
 nbformat.v4.new_markdown_cell('## 反復依頼履歴：反復3\n\n### 追加依頼文（原文）\n'+request3+'\n\n### 選定理由\n反復2で喫煙区分による率と会計額の順位関係に差（Yes約−0.480、No約−0.187）がある。群差を一定とするモデルから「喫煙区分は関係しない」と過剰解釈するのを避ける。証拠はiteration2セル、追加結果はiteration3セル。'),
 nbformat.v4.new_code_cell(iteration3,metadata={'analysis_id':'iteration3','chart':{'title':'喫煙区分と会計額の交互作用・感度分析','xlabel':'傾き差（通貨比/pp毎通貨）','ylabel':'分析仕様','legend':'95% HC3区間/傾き差ゼロ','missing_glyphs':[]}})
])
print('反復3を保存しました。')

反復3を保存しました。


In [17]:
summary = '''import importlib.metadata as package_metadata
with phase('capability_checks_and_reproducibility'):
    # 本実験で使う不確実性・図監査に対する限定した再現例。
    inferred_in_unresolved=any(path=='variables.total_bill.unit' for path,_ in definition_manifest.unresolved_fields())
    image_without_metadata=nbformat.v4.new_code_cell('render_chart(...)')
    image_without_metadata.outputs=[visualization.build_image_output(png)]
    mini_notebook=nbformat.v4.new_notebook(cells=[image_without_metadata])
    audit_without_metadata=notebook_audit.audit_visual_outputs(mini_notebook,(0,))
    capability_log={'inferred_unit_status':definition_manifest.variables['total_bill']['unit'].status,
      'inferred_unit_returned_by_unresolved_fields':inferred_in_unresolved,
      'visual_findings_when_chart_metadata_entirely_absent':[asdict(f) for f in audit_without_metadata]}
    emit({'capability_reproduction_log':capability_log})
    summary_metrics={**initial_metrics,**iteration1_metrics,**iteration2_metrics,**iteration3_metrics,
      'size_tip_pearson':float(correlations.loc[(correlations.feature=='size')&(correlations.outcome=='tip'),'Pearson_r'].iloc[0]),
      'adjusted_size_tip_CI95':fits['tip'].conf_int().loc['size'].tolist(),
      'adjusted_tip_bill_CI95':fits['tip'].conf_int().loc['total_bill'].tolist(),
      'tip_interaction_difference':float(interaction_table.loc[(interaction_table.outcome=='tip')&(interaction_table['subset']=='all'),'Yes_minus_No_bill_slope'].iloc[0]),
      'tip_interaction_CI95':interaction_table.loc[(interaction_table.outcome=='tip')&(interaction_table['subset']=='all'),['CI95_low','CI95_high']].iloc[0].tolist()}
    # 数値のfingerprintは主要統計・全係数・区間・感度仕様・bootstrap区間を含む。
    numeric_snapshot={'summary':summary_metrics,
      'coefficients':{k:coefficient_table(f).to_dict() for k,f in fits.items()},
      'sensitivity':sensitivity_table.to_dict('records'),'bootstrap':contrast_table.to_dict('records'),
      'interactions':interaction_table.to_dict('records')}
    baseline_file=data_dir/'analysis-baseline.json'
    def numeric_leaves(value,path=''):
        leaves={}
        if isinstance(value,dict):
            for k,v in value.items(): leaves.update(numeric_leaves(v,path+'.'+str(k)))
        elif isinstance(value,list):
            for i,v in enumerate(value): leaves.update(numeric_leaves(v,path+'.'+str(i)))
        elif isinstance(value,(int,float)) and not isinstance(value,bool):
            leaves[path]=float(value)
        return leaves
    if baseline_file.exists():
        baseline=json.loads(baseline_file.read_text(encoding='utf-8'))
        expected=numeric_leaves(baseline['numeric_snapshot']);actual=numeric_leaves(numeric_snapshot)
        assert expected.keys()==actual.keys()
        deviations={key:abs(actual[key]-expected[key]) for key in actual}
        assert all(np.isclose(actual[k],expected[k],atol=1e-10,rtol=1e-10) for k in actual), '数値再現検証が失敗'
        assert baseline['data_sha256']==provenance['sha256']
        reproducibility={'mode':'初回分析baselineとの数値比較','numeric_values_checked':len(actual),
          'max_absolute_difference':max(deviations.values()),'atol':1e-10,'rtol':1e-10,
          'match':True,'data_sha256_match':True,'seed':SEED}
    else:
        baseline_file.write_text(json.dumps({'data_sha256':provenance['sha256'],'numeric_snapshot':numeric_snapshot},ensure_ascii=False,indent=2),encoding='utf-8')
        reproducibility={'mode':'初回baselineを保存。新しいカーネルで再実行して照合する','numeric_values_checked':len(numeric_leaves(numeric_snapshot)),'match':None,'seed':SEED}
    versions={'python':sys.version.split()[0]}
    for name in ['numpy','pandas','scipy','statsmodels','matplotlib','nbformat','kaggle','jupyter-mcp-server','mcp']:
        versions[name]=package_metadata.version(name)
    emit({'summary_metrics':summary_metrics,'reproducibility':reproducibility,'package_versions':versions})
'''
modules = '''## 使用したJupytermindモジュール

以下は直接importし、本Notebookのコードから直接呼び出したもののみ。間接importや内部の依存は含めない。

| モジュール（`ai_data_scientist.` 以下） | 関数・クラス・メソッド | 使用目的 |
|---|---|---|
| project_manager | resolve_project, ensure_notebook, ensure_data_dir, enqueue_write | 指定slug・固定rootの解決、データ置場、Notebook単一書込み |
| language_router | detect_language | 日本語の検出 |
| lifecycle | register_run, mark_execution_start/end, mark_write_start/end, is_cancel_requested, mark_failed, mark_completed, get_run_status, wait_for_quiescence | 実行/書込みフェーズ・キャンセル確認・終状態 |
| ingestion | SourceSpec, ingest | 固定CSV読込みと行上限 |
| eda | explore | 欠損・型・要約・カテゴリ件数 |
| stats_analysis | correlation | Pearson相関と日本語interpretation |
| data_definition | FieldValue, build_manifest, DataDefinitionManifest.unresolved_fields | 定義・単位・出所の確度を区別 |
| analysis_assumptions | Assumption, AnalysisAssumptionManifest, check_manifest | 分析前提・因果範囲・未解決警告 |
| data_quality | detect_anomalies | 値域・欠損・カテゴリ制約の意味的検査 |
| sensitivity | SensitivityPlan, run_sensitivity | 仕様変更の係数安定性（予算6/4、許容偏差25%） |
| visualization | render_chart, build_image_output | 日本語フォント付き図生成・画像MIME化 |
| insight_engine | extract_cited_value, record_insight | 実行出力から引用値を抽出し根拠付きInsight記録 |
| notebook_audit | audit_visual_outputs, audit_notebook | メタデータ欠落の限定再現・最終visual_audit=True監査 |

### 適用できない機能・使用しなかった機能
`clean_dataset`は欠損補完・削除を必要とせず、同値会計を維持するので未使用。`validate_anomalies`/`compare_datasets`は独立した第二データがないため未使用。CSVの同系統コピーを独立した根拠とはしない。因果推論・ランダム化検証は識別情報がないため適用不可。クラスタ化標準誤差は客/担当者IDがなく適用不可。欠損補完・ML/予測チューニングは今回の記述/関連分析の目的に不要。

Python側の`mcp_gateway.run_and_record`は、本セッションのツール呼出し用MCP clientをPythonカーネルへ注入する接続adapterが提供されていないため未使用。同じカーネルへの再入的な呼出しや偽clientは用いない。実行経路はホスト側のJupyter MCP `execute_cell`に限定し、Notebookの著者側の書込みは`enqueue_write`で行った。MCPによる実行出力保存はMCPサーバーの管理に任せた。この統合上の適用制約は本分析の数値的不具合ではない。
'''
improvements = '''## Jupytermind改善候補

ソフトウェアの変更は行わず、今回確認した限定再現例を記録する。

| 分類 | 再現条件 | 期待する挙動 | 実際の挙動 | 影響 | 回避策 | 関係するセル・ログ |
|---|---|---|---|---|---|---|
| 機能不足：不確実性の列挙 | `total_bill.unit`を`inferred`としてbuild_manifest後、unresolved_fieldsを呼ぶ | Skill手順が注意喚起対象とするunknownとinferredの両方を取得可能、または対象差を明示 | unknownのみ返り、inferredは含まれない | 呼出し結果だけで注意書きを作ると単位推測を見落とす | manifestのstatusを別途走査しinferred_fieldsを明示 | quality / summary（capability_reproduction_log）のinferred判定False |
| 監査機能不足：図メタデータが丸ごと欠落 | 日本語PNGのコードセルにchart metadataを一切付けず、audit_visual_outputsを呼ぶ | missing_glyphs検査情報やタイトル/軸/凡例監査情報の欠落を警告 | 図が十分なPNGサイズならfindingsは空 | メタデータなしでも文字・ラベル検査済みのように受け取れる | 本Notebookの全図セルにchart metadataを明示し、描画時Glyph警告を検査。目視も実施 | summaryのvisual_findings_when_chart_metadata_entirely_absent=[]、charts_initial/iteration1/2/3 |

### 切り分け
Kaggleの`dataset_list_files`に対するHTTP 403は外部API/アクセス側の問題で、Jupytermind障害の証拠ではない。初回の検索一覧を例外経路で保持しなかった点はこのNotebookの取得コード側の記録不足であり、Jupytermindモジュールではない（現コードは保持するよう訂正、初回一覧は不明と明記）。Jupyter MCPで未作成ディレクトリへのcreateが失敗した点はMCPサーバー側のパス前提で、project_manager.ensure_notebookで解消した。Copilot CLI・ベンチマークランナー固有の数値分析の問題は観測していない。Python gateway adapter未提供は統合制約で、今回の図監査/manifest再現とは別である。
'''
final = '''import re
with phase('evidence_and_handoff'):
    evidence_nb=nbformat.read(handle.notebook_path,as_version=4)
    evidence_cell=next(c for c in evidence_nb.cells if c.metadata.get('analysis_id')=='summary')
    evidence_text='\\n'.join(str(o.get('data',{}).get('text/plain','')) for o in evidence_cell.outputs)
    ec=evidence_cell.execution_count
    # 出力に存在する正確な数値を使い、再実行で変わったexecution_countも再解決する。
    citations={key:insight_engine.extract_cited_value({'output':evidence_text},r'"'+re.escape(key)+r'":\\s*([-+0-9.eE]+)')
               for key in ['mean_rate_pct','bill_tip_pearson','size_tip_pearson','bill_rate_pearson',
                           'rate_max_relative_deviation','smoker_mean_difference_pp','time_VIF','tip_interaction_difference']}
    def reset_generated(nb):
        nb.cells[:]=[c for c in nb.cells if not c.metadata.get('generated_insight')]
    write_notebook(reset_generated)
    def signed(value, digits=3):return f'{value:+.{digits}f}'
    m=summary_metrics
    narratives=[
      ('mean_rate_pct',f'全{m["n"]}会計の平均チップ額は{m["mean_tip"]:.3f}（原データ通貨単位）。会計ごとの平均チップ率は{m["mean_rate_pct"]:.3f}%、中央値は{m["median_rate_pct"]:.3f}%、金額加重率は{m["bill_weighted_rate_pct"]:.3f}%。平均率と金額加重率は同じ指標ではない。通貨・税・総額のチップ含有は辞書未確認なので、率は列定義上の値として解釈する。'),
      ('bill_tip_pearson',f'初回分析：会計総額とチップ額には正の関連（Pearson r={m["bill_tip_pearson"]:.3f}、Spearman rho={float(correlations.loc[(correlations.feature=="total_bill")&(correlations.outcome=="tip"),"Spearman_rho"].iloc[0]):.3f}）。他の観測特徴を一定とした初回モデルでも、会計額1単位に対してチップ額は{m["adjusted_tip_bill_coef"]:.3f}単位高い（HC3 95%区間 {m["adjusted_tip_bill_CI95"][0]:.3f}〜{m["adjusted_tip_bill_CI95"][1]:.3f}）。後の交互作用分析ではこの傾きが喫煙区分で異なるため、共通の因果効果として扱わない。'),
      ('size_tip_pearson',f'人数はチップ額と正に関連（r={m["size_tip_pearson"]:.3f}）するが、会計額などを調整した1人増加の係数は{m["adjusted_tip_size_coef"]:.3f}、95%区間は{m["adjusted_size_tip_CI95"][0]:.3f}〜{m["adjusted_size_tip_CI95"][1]:.3f}で0を含む。人数そのものの独立した関係は強く断定できない。1/5/6人群は4/5/4会計で、平均の単純な順位付けは不安定。'),
      ('bill_rate_pearson',f'会計総額とチップ率には負の観察上の関連（Pearson r={m["bill_rate_pearson"]:.3f}、Spearman rho={m["bill_rate_spearman"]:.3f}）。対数モデルのチップ額弾力性は{m["elasticity"]:.3f}（95%区間 {m["elasticity_CI95"][0]:.3f}〜{m["elasticity_CI95"][1]:.3f}）。これは観測範囲のモデル上、額が会計額に比例しては増えていないことと整合する。ただしtip/billはbillを分母に共有し、加法モデルの正の定額成分だけでも率は下がる。「高額な客ほど気前が悪い」「会計額を増やすと率が下がる」といった心理・因果の結論にはならない。'),
      ('rate_max_relative_deviation',f'反復1：6仕様で会計額とチップ額の傾きはすべて正、最大相対偏差{100*m["tip_max_relative_deviation"]:.2f}%、stable={m["tip_sensitivity_stable"]}。率の傾きはすべて負だが最大相対偏差{100*m["rate_max_relative_deviation"]:.2f}%で25%基準を超え、stable={m["rate_sensitivity_stable"]}。率の向きは維持されたが大きさは頑健と言えない。証拠はiteration1とsummary。残る限界は仕様・部分集合による推定対象の違いと比率の構造。'),
      ('smoker_mean_difference_pp',f'反復2：喫煙Yes−Noの会計平均率差は{signed(m["smoker_mean_difference_pp"])}pp（bootstrap 95%区間 {m["smoker_mean_CI95"][0]:.3f}〜{m["smoker_mean_CI95"][1]:.3f}）だが、金額加重率差は{signed(m["smoker_weighted_difference_pp"])}ppと逆向き。大きな会計を重くするかによって推定対象が変わり、Yes群のbillとrateの負の共変動が強いため反転する。これは喫煙の効果の符号が変わったという意味ではない。証拠はiteration2とsummary。残る限界は群定義未確認・観測交絡・独立性・探索的な区間。'),
      ('time_VIF',f'反復2：曜日×時間帯は土日が夕食のみ、木曜夕食は1会計。昼夜係数のVIFは{m["time_VIF"]:.3f}で、同じ曜日の比較情報が乏しい。金曜の昼7/夜12会計だけの率差は{signed(m["Friday_lunch_dinner_difference_pp"])}ppだが95%区間は{m["Friday_lunch_dinner_CI95"][0]:.3f}〜{m["Friday_lunch_dinner_CI95"][1]:.3f}。曜日・昼夜・記録上の性別の初回調整区間は0を含むが、「関係がない」「同等」とは証明できない。証拠はdescriptive/models/iteration2。'),
      ('tip_interaction_difference',f'反復3：会計額×喫煙区分の交互作用を探索すると、チップ額のYes−No傾き差は{signed(m["tip_interaction_difference"])}（95%区間 {m["tip_interaction_CI95"][0]:.3f}〜{m["tip_interaction_CI95"][1]:.3f}）。4仕様で負の傾き差を保ち、最大相対偏差{100*m["tip_interaction_max_relative_deviation"]:.2f}%、stable={m["tip_interaction_stable"]}。一方、率の交互作用は偏差{100*m["rate_interaction_max_relative_deviation"]:.2f}%、stable={m["rate_interaction_stable"]}で、高率上位1%除外の95%区間は0を含む。群全体の平均差が小さくても金額との関係は異なり得る、という探索的候補に限定する。証拠はiteration3とsummary。事後選択・多重比較・交絡・会計間依存の限界が残る。')
    ]
    for key,text in narratives:
        with phase('record_insight',writing=True):
            insight_engine.record_insight(handle,text+'\n\n根拠の正確な引用値: `'+citations[key]+'`。',ec,citations[key],'associational',language='ja')
        def tag(nb):nb.cells[-1].metadata['generated_insight']=True
        write_notebook(tag)
    def organize(nb):
        last=next(c for c in nb.cells if c.metadata.get('analysis_id')=='final_audit')
        nb.cells.remove(last);nb.cells.append(last)
    write_notebook(organize)
    print('Insightを実行済み出力から再生成しました。引用先execution_count=',ec)
# 自己監査のため、未完了の末尾セルは監査器が除外する。
with phase('final_notebook_audit'):
    audit=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    if not audit.ok:
        emit({'audit_failure':asdict(audit)})
        raise RuntimeError('Notebook監査に未解決の指摘があります')
lifecycle.mark_completed(RUN_ID)
status=lifecycle.wait_for_quiescence(RUN_ID,timeout_s=5)
assert status.state=='completed'
assert status.active_cell_executions==status.pending_notebook_writes==status.locks_held==0
with phase('persist_lifecycle_report',writing=True):
    (data_dir/'lifecycle-final.json').write_text(json.dumps({'run_status':asdict(status),'audit':asdict(audit),'reproducibility':reproducibility},ensure_ascii=False,indent=2),encoding='utf-8')
emit({'notebook_audit':asdict(audit),'audit_ok':audit.ok,'lifecycle':asdict(lifecycle.get_run_status(RUN_ID)),
      'event_log':event_log,'reproducibility':reproducibility,
      'handoff':'3反復で新たな解釈上の論点を確認。残る課題はID・辞書・独立検証データが必要で、同じデータ内の追加探索では解消できない。'})
'''
append_cells([
 nbformat.v4.new_markdown_cell('## 限定再現例・数値再現チェック\n初回実行で数値baselineを保存し、最後の新カーネル再実行で同一SHA-256/seed・主要数値と区間を照合する。API成功を擬装せず、認証/検索失敗の記録と固定ファイルによる分析再現を区別する。'),
 nbformat.v4.new_code_cell(summary,metadata={'analysis_id':'summary'}),
 nbformat.v4.new_markdown_cell(modules),nbformat.v4.new_markdown_cell(improvements),
 nbformat.v4.new_markdown_cell('## 根拠付きInsightと反復結果\n以下はsummaryセルの実行済み出力から引用値を抽出して記録する。すべて観察上の関連であり、カテゴリについて無効果や同等性を証明するものではない。\n\n### 反復の終了判断\n最大3回の追加依頼を実行した。外れ値/分母共有、群差と重なり不足、交互作用という主結論に影響する論点を検討した。残る限界はデータ辞書未確認、独立性不明、未観測交絡、事後探索、第二データ不在であり、同じ244会計への追加分析を増やしても識別できない。確証には辞書・客/担当者ID・別期間/店のデータが必要。'),
 nbformat.v4.new_markdown_cell('## 最終監査・ライフサイクル\n再実行で根拠manifestのexecution_countを再生成し、図・エラー・未実行・引用を監査する。完了時はactive execution/pending write/locksが全て0であることを確認する。'),
 nbformat.v4.new_code_cell(final,metadata={'analysis_id':'final_audit'})
])
print('最終記録・再実行検証・監査コードを保存しました。')

最終記録・再実行検証・監査コードを保存しました。


In [20]:
import ast
def repair_final(nb):
    cell=next(c for c in nb.cells if c.metadata.get('analysis_id')=='final_audit')
    cell.source=cell.source.replace("text+'\n\n根拠", "text+'\\n\\n根拠")
    cell.outputs=[]
    cell.execution_count=None
    for c in nb.cells:
        if c.cell_type=='code':ast.parse(c.source)
with phase('repair_notebook_authoring_escape',writing=True):
    project_manager.enqueue_write(handle,repair_final)
print('Notebook著者側の改行エスケープを修正し、全コードの構文を確認しました。')

Notebook著者側の改行エスケープを修正し、全コードの構文を確認しました。


In [23]:
def prepare_clean_replay(nb):
    final_cell=next(c for c in nb.cells if c.metadata.get('analysis_id')=='final_audit')
    final_cell.source=final_cell.source.replace("    def reset_generated(nb):\n        nb.cells[:]=[c for c in nb.cells if not c.metadata.get('generated_insight')]\n    write_notebook(reset_generated)",
        "    generated_positions=[i for i,c in enumerate(evidence_nb.cells) if c.metadata.get('generated_insight')]\n    assert len(generated_positions)==8, 'Insightの固定記録枠が不足'")
    final_cell.source=final_cell.source.replace("    for key,text in narratives:", "    for slot,(key,text) in zip(generated_positions,narratives,strict=True):")
    final_cell.source=final_cell.source.replace("        def tag(nb):nb.cells[-1].metadata['generated_insight']=True", "        def tag(nb):\n            generated=nb.cells.pop()\n            generated.metadata['generated_insight']=True\n            nb.cells[slot]=generated")
    note=next(c for c in nb.cells if c.cell_type=='markdown' and c.source.startswith('## Jupytermind改善候補'))
    note.source+='\n\nNotebook著者側の初回コード生成で改行エスケープ不足のSyntaxErrorを修正した（final_audit、以後全コードを構文確認）。また初回のInsight挿入で実行中セルの位置が24→32へ変わり、MCPのindexベース出力保存が追従せず最終セル出力が未保存となった。これはセル移動とMCP実行の統合問題で、Jupytermind分析モジュールの不具合ではない。回避策はInsight記録枠を固定し、再実行時は同じ位置へ差替え、実行中セル位置を変えない。これらは最終の新カーネル再実行で再検証する。'
    nb.metadata['v020_exp_04']={'run_id':RUN_ID,'replay_protocol':'Jupyter MCP kernel restart; execute all code cells in ascending index', 'expected_code_cells':11}
    for c in nb.cells:
        if c.cell_type=='code':
            ast.parse(c.source)
            c.outputs=[];c.execution_count=None
with phase('prepare_clean_top_to_bottom_replay',writing=True):
    project_manager.enqueue_write(handle,prepare_clean_replay)
print('全11コードセルの出力をクリアしました。新カーネルで順次実行します。')

全11コードセルの出力をクリアしました。新カーネルで順次実行します。
